In [1]:
!pip3 install -r requirements.txt

  Cloning https://github.com/evfro/polara.git to /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-vktiaise/polara_d605fdc0d6574cfebcbe359b4fbffd4a
  Running command git clone --filter=blob:none --quiet https://github.com/evfro/polara.git /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-vktiaise/polara_d605fdc0d6574cfebcbe359b4fbffd4a
  Resolved https://github.com/evfro/polara.git to commit 86a5d247335242f31baf8fb68e472b651ae6770f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 735.1 kB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 885.2 kB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [2]:
from typing import Callable, Dict, List, Tuple, Any, Optional
from collections import defaultdict


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm


import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch.nn.functional as F



from src.data.dataprep import (transform_indices, verify_time_split, reindex_data, temporal_train_test_split)
from src.data.loaders import (load_ml20m, load_amzn_books, load_yambda, split_and_reindex)
from src.data.evaluation import (evaluate)
from src.data.utils import (create_masked_tensor, build_q_from_train_targets)

In [3]:
class TransfromerTrainDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        self.samples = []

        for uid, history in histories.items():
            indicies = np.arange(0, len(history), max_seq_len)
            splits = np.split(history[:-1], indicies)[1:]
            targets = np.split(history, indicies+1)[1:]
            
            self.samples.extend(
                {
                    'uid': uid,
                    'history': list(s),
                    'targets': list(t),
                    'length': len(s)
                }
                for s, t in zip(splits, targets) if len(s) > 0 and len(t) > 0
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [4]:
class TransfromerEvalDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        targets: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        
        targets_uids = targets.keys()
        self.samples = [
            {
                'uid': uid,
                'history': history[-max_seq_len:],
                'length': len(history[-max_seq_len:])
            }
            for uid, history in histories.items() if uid in targets_uids and history
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [5]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:

    uids = torch.stack([torch.tensor(b['uid'], dtype=torch.long) for b in batch])
    lengths = torch.stack([torch.tensor(b['length'], dtype=torch.long) for b in batch])
    histories = torch.concat([torch.tensor(b['history'], dtype=torch.long) for b in batch])

    res = {
        'uid': uids,
        'length': lengths,
        'history': histories
        }

    if batch[0].get('targets', None):
        targets = torch.concat([torch.tensor(b['targets'], dtype=torch.long) for b in batch])
        res['targets'] = targets

    return res

In [6]:
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128
DEVICE='cpu'

config = {
    # "ml": {
    #     "max_seq_len": 200,
    #     "test_quantile": 0.1,
    #     "interactions_path": "ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
    #     "col_mapping": {
    #         "userid": "user_id",
    #         "movieid": "item_id",
    #         "rating": "feedback",
    #         "timestamp": "timestamp",
    #     },
    # },
    "yambda": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "yambda-10m.parquet", # из ноута Кирилла (мб кто подкинет ссылку)
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    # "amzn-books": {
    #     "max_seq_len": 50,
    #     "test_quantile": 0.1,
    #     "interactions_path": "amzn-books-20m.csv.gz", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
    #     "col_mapping": {
    #         "user_id": "user_id",
    #         "parent_asin": "item_id",
    #         "rating": "feedback",
    #         "timestamp": "timestamp",
    #     },
    # },
}

In [7]:
# ml_df = load_ml20m(config["ml"]["interactions_path"], config=config["ml"])
yambda_df = load_yambda(config["yambda"]["interactions_path"], config=config["yambda"])
# amzn_df = load_amzn_books(config["amzn-books"]["interactions_path"], config=config["amzn-books"])

In [8]:
# ml_train, ml_test, ml_data_description = split_and_reindex(ml_df, config=config["ml"])
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda"])
# amzn_train, amzn_test, amzn_data_description = split_and_reindex(amzn_df, config=config["amzn-books"])

In [9]:
datasets = {
    # "ml": {
    #     "train": ml_train,
    #     "test": ml_test,
    #     "desc": ml_data_description,
    # },
    "yambda": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    },
    # "amzn-books": {
    #     "train": amzn_train,
    #     "test": amzn_test,
    #     "desc": amzn_data_description,
    # },
}

In [10]:
def build_histories(df: pd.DataFrame, desc: Dict) -> Dict:
    return df.sort_values([desc['users'], desc['timestamp']], ascending=True).groupby(desc['users'])[desc['items']].apply(list).to_dict()

for name, d in datasets.items():
    train, test, desc = d['train'], d['test'], d['desc']
    max_seq_len = config[name]['max_seq_len']

    histories = build_histories(train, desc)
    targets = build_histories(test, desc)
    
    d['histories'] = histories
    d['targets'] = targets  

    d['train_dataset'] = TransfromerTrainDataset(histories=histories, max_seq_len=max_seq_len)
    d['eval_dataset'] = TransfromerEvalDataset(histories=histories, targets=targets, max_seq_len=max_seq_len)

    d['train_dataloader'] = DataLoader(
        dataset=d['train_dataset'],
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        drop_last=True
    )
    d['eval_dataloader'] = DataLoader(
        dataset=d['eval_dataset'],
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        drop_last=False
    )

In [11]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,yambda,7473676,748537,8222213,0.909,0.091,79059,53927,163448


TopPop

In [12]:
class TopPopRec:
    def __init__(self, n_items: int):
        self.n_items = n_items
        self.popular_items = None

    def fit(self, train_histories: Dict[int, List[int]]):
        all_items = np.concatenate(list(train_histories.values()))
        counts = np.bincount(all_items, minlength=self.n_items)
        self.popular_items = np.argsort(counts)[::-1].tolist()
        return self

    def predict(self, user_ids: List[int], topk: int = 100) -> Dict[int, List[int]]:
        top = self.popular_items[:topk]
        return dict(zip(user_ids, [top] * len(user_ids)))

SASRec

In [13]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.dropout = dropout

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.n_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, n_heads, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)


    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:

        x = self.qkv(x)

        Q = self.split_heads(x[:, :, :self.d_model])
        K = self.split_heads(x[:, :, self.d_model:2*self.d_model])
        V = self.split_heads(x[:, :, 2*self.d_model:])

        causal_mask = torch.ones(Q.size(-2), K.size(-2), dtype=torch.bool, device=Q.device).tril(diagonal=0).unsqueeze(0).unsqueeze(0)
        padding_mask = mask.unsqueeze(1).unsqueeze(1)
        attn_mask = causal_mask & padding_mask

        attn_output = F.scaled_dot_product_attention(Q, K, V, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0)
        return self.out(self.combine_heads(attn_output))

In [14]:
class MLP(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.0):
        super().__init__()
        self.linear = nn.Linear(d_model, 4 * d_model)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(4 * d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, d_model = x.size()

        x = self.linear(x)
        x = self.gelu(x)
        x = self.dropout(x)
        x = self.out(x)

        return x

In [15]:
class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()

        self.dropout = dropout

        self.ln1 = nn.LayerNorm(d_model)
        self.csa = CausalSelfAttention(d_model, n_heads, dropout)
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:

        B, S, D = x.size()

        if mask is None:
            mask = torch.ones(B, S, dtype=torch.bool, device=x.device)

        x1 = self.ln1(x)
        x1 = self.csa(x1, mask)
        x1 = self.dropout1(x1)

        x = x + x1

        x2 = self.ln2(x)
        x2 = self.mlp(x2)
        x2 = self.dropout2(x2)

        x = x + x2


        return x

In [16]:
class GPT(nn.Module):
    def __init__(
        self,
        max_seq_len: int,
        n_layers: int,
        d_model: int,
        n_heads: int,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.dropout = dropout
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:

        x = self.drop(x)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln(x)

        return x

In [17]:
class UserEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.item_embeddings = nn.Embedding(num_items, embedding_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embedding_dim)
        self.GPT = GPT(max_seq_len, n_layers, embedding_dim, n_heads, dropout)



    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:

        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
        item_emb = self.item_embeddings(hist)
        pos_emb = self.pos_emb(torch.arange(hist.size(1), dtype=torch.long, device=hist.device))
        x = item_emb + pos_emb
        x[~mask] = 0.0
        x = self.GPT(x, mask)

        return x[mask]

In [18]:
class TrainSASRecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q_counts: torch.Tensor,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        
        self.encoder = UserEncoder(
            num_items=num_items,
            embedding_dim=embedding_dim,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            n_heads=n_heads,
            dropout=dropout,
        )
        self.num_negatives = num_negatives
        self.eps = eps

        q_counts = q_counts.detach().float().clamp_min(0)
        self.register_buffer("q_counts", q_counts)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        for key, value in self.named_parameters():
            if "weight" in key:
                nn.init.trunc_normal_(
                    value.data,
                    std=initializer_range,
                    a=-2 * initializer_range,
                    b=2 * initializer_range,
                )
            elif "bias" in key:
                nn.init.zeros_(value.data)


    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        return self.compute_loss(encoder_output, inputs)


    def compute_loss(
        self, encoder_output: torch.Tensor, inputs: Dict[str, Any]
    ) -> torch.Tensor:

        target_ids = inputs['targets']
        n = target_ids.shape[0]

        negative_pos = torch.randint(0, n, (n, self.num_negatives), device=target_ids.device)
        negative_ids = target_ids[negative_pos]

        candidate_ids = torch.cat([target_ids[:, None], negative_ids], dim=1)
        candidate_emb = self.encoder.item_embeddings(candidate_ids)

        logits = (encoder_output[:, None, :] * candidate_emb).sum(dim=-1)

        log_total = torch.log(self.q_counts.sum().clamp_min(self.eps))
        log_q = torch.log(self.q_counts[negative_ids].clamp_min(self.eps)) - log_total

        correction = torch.cat([torch.zeros(n, 1, device=logits.device), log_q], dim=1)
        logits = logits - correction

        labels = torch.zeros(n, dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

In [19]:
def train(
    dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    device: str = "cuda",
) -> tuple[dict[str, Any], list[float]]:

    model.train()
    epoch_losses = []

    for epoch in tqdm(range(num_epochs), desc="Epochs"):
        total_loss = 0.0
        num_batches = 0

        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            loss = model(batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        epoch_loss = total_loss / num_batches
        epoch_losses.append(epoch_loss)
        tqdm.write(f"Epoch {epoch+1} loss: {epoch_loss:.4f}")

    return model.state_dict(), epoch_losses

In [20]:
class EvalSASRecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        
        self.encoder = UserEncoder(
            num_items=num_items,
            embedding_dim=embedding_dim,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            n_heads=n_heads,
            dropout=dropout
        )


    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        lengths = inputs['length']
        last_idx = lengths.cumsum(dim=0) - 1
        user_emb = encoder_output[last_idx]

        item_emb = self.encoder.item_embeddings.weight
        scores = user_emb @ item_emb.T

        return scores

In [ ]:
results = []

for name, d in datasets.items():
    desc = d['desc']
    user_ids = list(d['targets'].keys())

    # TopPop
    toppop_model = TopPopRec(n_items=desc['n_items']).fit(d['histories'])
    toppop_recs = toppop_model.predict(user_ids, topk=10)
    toppop_metrics = evaluate(
        targets=d['targets'],
        candidates=toppop_recs,
        catalog_size=desc['n_items'],
        topk=10
    )
    print(f"[{name}] TopPop metrics: {toppop_metrics}")
    results.append({'dataset': name, 'model': 'toppop', **toppop_metrics})

    # SASRec
    train_target_ids = torch.tensor(
        [target for idx in range(len(d['train_dataset'])) for target in d['train_dataset'][idx]['targets']],
        dtype=torch.long,
    )
    q_counts = build_q_from_train_targets(train_target_ids, catalog_size=desc['n_items'])

    train_model = TrainSASRecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        num_negatives=512,
        q_counts=q_counts,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
        dropout=0.1,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(train_model.parameters(), lr=1e-3)

    checkpoint, losses = train(
        dataloader=d['train_dataloader'],
        model=train_model,
        optimizer=optimizer,
        num_epochs=10,
        device=DEVICE,
    )

    eval_model = EvalSASRecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
    ).to(DEVICE)

    encoder_state = {
        key.removeprefix("encoder."): value
        for key, value in checkpoint.items()
        if key.startswith("encoder.")
    }
    eval_model.encoder.load_state_dict(encoder_state)

    metrics = eval(
        dataloader=d['eval_dataloader'],
        model=eval_model,
        catalog_size=desc['n_items'],
        topk=10,
        device=DEVICE,
        targets=d['targets'],
        evaluate_fn=evaluate,
    )

    print(f"[{name}] SASRec metrics: {metrics}")
    results.append({'dataset': name, 'model': 'sasrec', **metrics})

pd.DataFrame(results)

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/705 [00:00<?, ?it/s]

KeyboardInterrupt: 